In [1]:
from   ray.tune.schedulers import ASHAScheduler
import ray.cloudpickle as pickle
from   ray import tune
from   ray import train
# from ray.train import Checkpoint, get_checkpoint


In [2]:
from train import GenerateModel, federate_model
from torch.utils.data import DataLoader, TensorDataset
from metrics import eval_model
from functools import partial
import torch
import os

In [3]:
#working_dir = os.path.split(os.getcwd())[0]
working_dir  = "/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS" 
X_test = torch.load(os.path.join(working_dir,"Data","X_test.pt"))
Y_test = torch.load(os.path.join(working_dir,"Data","Y_test.pt"))

# model = GenerateModel(table_path     = os.path.join(working_dir,"Model","processed_full.w2v"),
#                       num_of_filters = 15,
#                       kernel_size    = 5)

def load_data():
    # working_dir = os.path.split(os.getcwd())[0]
    working_dir  = "/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS" 
    X_val  = torch.load(os.path.join(working_dir,"Data","X_val.pt"))
    Y_val  = torch.load(os.path.join(working_dir,"Data","Y_val.pt"))
    return DataLoader(TensorDataset(X_val,Y_val),batch_size=32,shuffle=False)

In [4]:
# val_loader = load_data()

In [5]:
def train_model(config):
    """
    config = {
        "batch_size" : 16,
        "lr"         : 0.0001,
        "n_filters"  : 5,
        "window_size": 3,
        "epochs"     : 2,
        "rounds"     : 5
    }
    """
    trained_model = federate_model(config)
    working_dir  = "/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS" 
    model = GenerateModel(table_path = os.path.join(working_dir,"Model","processed_full.w2v"),
                      num_of_filters = config['n_filters'],
                      kernel_size    = config['window_size'])
    model.load_state_dict(trained_model)
    ########################################
    saved_metric = eval_model(
               model       = model,
               device      = torch.device("cpu"),
               data_loader = load_data() # val_loader
               )
    print(saved_metric)
    ########################################
    tune.report(saved_metric)

In [6]:
# config = {
#     "l1": tune.choice([10,15,20]), # Num of Filters
#     "l2": tune.choice([3,4,5]),    # Filtern size
#     "lr": tune.loguniform(0.0001, 0.1),
#     "batch_size": tune.choice([8, 16, 32]),
# }

config = {
        "batch_size" : tune.choice([8, 16, 32]),
        "lr"         : tune.loguniform(0.0001, 0.1),
        "n_filters"  : tune.choice([10,15,20]),
        "window_size": tune.choice([3,4,5]),
        "epochs"     : tune.choice([2,3,5]),
        "rounds"     : 2
}
scheduler = ASHAScheduler(
    metric = "auc_macro",
    mode   = "max",
    max_t  = 600, # Max time in seconds
    grace_period     = 100,
    reduction_factor = 2
)

In [7]:
result = tune.run(
    partial(train_model),
    resources_per_trial={"cpu": 2, "gpu": 1},
    config      = config,
    num_samples = 5, # Number of different combinations
    scheduler   = scheduler,
    storage_path="/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS/S3_Bucket/results"
    )

2025-09-09 05:25:12,926	INFO worker.py:1927 -- Started a local Ray instance.
2025-09-09 05:25:13,916	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `tune.run(...)`.
2025-09-09 05:25:13,917	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


(func pid=18141) {'acc_macro': 0.16526635724448205, 'prec_macro': 0.17046649609636638, 'rec_macro': 0.9313093561613164, 'f1_macro': 0.2881839212599706, 'acc_micro': 0.14974035924065718, 'prec_micro': 0.15079297042434633, 'rec_micro': 0.955458989679522, 'f1_micro': 0.26047682511476383, 'rec_at_8': 0.5208651546095572, 'prec_at_8': 0.33989460632362056, 'f1_at_8': 0.41135579219403373, 'auc_macro': 0.7670999864458383, 'auc_micro': 0.7891911309068116}


Trial name,acc_macro,acc_micro,auc_macro,auc_micro,f1_at_8,f1_macro,f1_micro,prec_at_8,prec_macro,prec_micro,rec_at_8,rec_macro,rec_micro
train_model_5ce13_00000,0.165266,0.14974,0.7671,0.789191,0.411356,0.288184,0.260477,0.339895,0.170466,0.150793,0.520865,0.931309,0.955459
train_model_5ce13_00001,0.135674,0.133155,0.811418,0.844485,0.484859,0.238865,0.235017,0.398636,0.135741,0.133219,0.618677,0.994062,0.996415
train_model_5ce13_00002,0.170566,0.157656,0.792574,0.849317,0.479676,0.291282,0.27237,0.395381,0.171483,0.158269,0.609652,0.96645,0.975991
train_model_5ce13_00003,0.181112,0.167416,0.806672,0.860281,0.491245,0.306325,0.286814,0.404758,0.182135,0.168127,0.624736,0.96286,0.975339
train_model_5ce13_00004,0.122556,0.123118,0.859457,0.891575,0.539731,0.218341,0.219243,0.443351,0.122562,0.123126,0.689654,0.999161,0.999457


(func pid=18351) {'acc_macro': 0.1356739332690363, 'prec_macro': 0.13574144067362043, 'rec_macro': 0.9940621449428001, 'f1_macro': 0.23886528488936284, 'acc_micro': 0.13315525100897185, 'prec_micro': 0.13321907362488924, 'rec_micro': 0.9964149918522542, 'f1_micro': 0.23501678325262, 'rec_at_8': 0.6186766000969579, 'prec_at_8': 0.3986360818350899, 'f1_at_8': 0.48485941474222133, 'auc_macro': 0.8114175772934916, 'auc_micro': 0.8444847236466411}
(func pid=18547) {'acc_macro': 0.17056637403705316, 'prec_macro': 0.1714828055377111, 'rec_macro': 0.9664500448028881, 'f1_macro': 0.29128180111021607, 'acc_micro': 0.15765552338334649, 'prec_micro': 0.15826932562891974, 'rec_micro': 0.9759913090711569, 'f1_micro': 0.27237035577316615, 'rec_at_8': 0.6096518242374568, 'prec_at_8': 0.3953812771233726, 'f1_at_8': 0.47967557793115717, 'auc_macro': 0.7925739910985292, 'auc_micro': 0.8493174410189839}
(func pid=18707) {'acc_macro': 0.181111866512632, 'prec_macro': 0.18213508249774363, 'rec_macro': 0.962

2025-09-09 05:27:36,985	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS/S3_Bucket/results/train_model_2025-09-09_05-25-13' in 0.0150s.


(func pid=18894) {'acc_macro': 0.12255568056971608, 'prec_macro': 0.12256160986970645, 'rec_macro': 0.9991614064338754, 'f1_macro': 0.21834058624517647, 'acc_micro': 0.12311809969889595, 'prec_micro': 0.12312633832976445, 'rec_micro': 0.9994568169473113, 'f1_micro': 0.21924337205838546, 'rec_at_8': 0.6896537930247448, 'prec_at_8': 0.4433508989460632, 'f1_at_8': 0.5397305611633967, 'auc_macro': 0.8594570600727204, 'auc_micro': 0.8915754935300241}


2025-09-09 05:27:36,996	INFO tune.py:1041 -- Total run time: 143.08 seconds (143.03 seconds for the tuning loop).
